# Collect Demonstration from Keyboard

Collect demonstration data for the bimanual FFW robot.
The task is to pick a mug and place it on the plate. The environment recognizes success if the mug is on the plate, the gripper is open, and the end-effector is positioned above the mug.

<img src="./media/teleop.gif" width="480" height="360">

## Keyboard Controls

**LEFT ARM** (`arm_l_link7`)
| Key | Action |
|-----|--------|
| W / S | ± X (forward / backward) |
| A / D | ± Y (left / right) |
| R / F | ± Z (up / down) |
| Q / E | Yaw (rotate around Z) |
| ↑ / ↓ | Pitch (rotate around X) |
| ← / → | Roll (rotate around Y) |
| SPACE | Toggle left gripper |

**RIGHT ARM** (`arm_r_link7`)
| Key | Action |
|-----|--------|
| I / K | ± X (forward / backward) |
| J / L | ± Y (left / right) |
| U / O | ± Z (up / down) |
| , / . | Yaw (rotate around Z) |
| HOME / END | Pitch (rotate around X) |
| PGUP / PGDN | Roll (rotate around Y) |
| ENTER | Toggle right gripper |

**General**
| Key | Action |
|-----|--------|
| Z | Reset episode (discards current data) |

## Viewer Overlays
- **Top Right**: Agent View
- **Bottom Right**: Top View
- **Top Left**: Side View (teleop mode only)

In [ ]:
import os  # if working on remote, set the display number
os.environ['DISPLAY']=':5'

In [ ]:
import sys
import random
import numpy as np
import os
from PIL import Image
from mujoco_env.y_env import SimpleEnv
from lerobot.common.datasets.lerobot_dataset import LeRobotDataset

In [ ]:
# If you want to randomize the object positions, set this to None
# If you fix the seed, the object positions will be the same every time
SEED = 0 
# SEED = None <- Uncomment this line to randomize the object positions

REPO_NAME = 'ffw'
NUM_DEMO = 1 # Number of demonstrations to collect
ROOT = "./ffw_demo_data" # The root directory to save the demonstrations

In [ ]:
TASK_NAME = 'Put mug cup on the plate'
xml_path = './asset/example_scene_y.xml'
# Define the environment
# action_type='eef_pose': teleop sends EEF delta poses, IK runs internally
# state_type='joint_angle': observations returned as joint angles (16D)
PnPEnv = SimpleEnv(xml_path, seed=SEED, action_type='eef_pose', state_type='joint_angle')

## Define Dataset Features and Create your dataset!

Teleop drives the robot via EEF delta poses → IK → joint angles. Both observations and actions are recorded as joint angles (16D: 14 arm joints + 2 gripper states).

```
fps = 20,
features={
    "observation.images.agentview": {
        "dtype": "image",
        "shape": (256, 256, 3),
        "names": ["height", "width", "channels"],
    },
    "observation.images.topview": {
        "dtype": "image",
        "shape": (256, 256, 3),
        "names": ["height", "width", "channels"],
    },
    "observation.images.sideview": {
        "dtype": "image",
        "shape": (256, 256, 3),
        "names": ["height", "width", "channels"],
    },
    "observation.state": {
        "dtype": "float32",
        "shape": (16,),
        "names": ["state"],  # current joint angles: 7L arm + 7R arm + 2 gripper states
    },
    "action": {
        "dtype": "float32",
        "shape": (16,),
        "names": ["action"],  # target joint angles from IK: 7L arm + 7R arm + 2 gripper cmds
    },
    "obj_init": {
        "dtype": "float32",
        "shape": (6,),
        "names": ["obj_init"],  # initial positions: mug (3) + plate (3). Not used in training.
    },
},
```

Dataset directory structure:
```
.
├── data
│   └── chunk-000
│       ├── episode_000000.parquet
│       └── ...
└── meta
    ├── episodes.jsonl
    ├── info.json
    ├── stats.json
    └── tasks.jsonl
```

In [ ]:
create_new = True
if os.path.exists(ROOT):
    print(f"Directory {ROOT} already exists.")
    ans = input("Do you want to delete it? (y/n) ")
    if ans == 'y':
        import shutil
        shutil.rmtree(ROOT)
    else:
        create_new = False

if create_new:
    dataset = LeRobotDataset.create(
        repo_id=REPO_NAME,
        root=ROOT,
        robot_type="ffw",
        fps=20,
        features={
            "observation.images.agentview": {
                "dtype": "image",
                "shape": (256, 256, 3),
                "names": ["height", "width", "channels"],
            },
            "observation.images.topview": {
                "dtype": "image",
                "shape": (256, 256, 3),
                "names": ["height", "width", "channels"],
            },
            "observation.images.sideview": {
                "dtype": "image",
                "shape": (256, 256, 3),
                "names": ["height", "width", "channels"],
            },
            "observation.state": {
                "dtype": "float32",
                "shape": (16,),
                "names": ["state"],  # 7L arm + 7R arm + 2 gripper states
            },
            "action": {
                "dtype": "float32",
                "shape": (16,),
                "names": ["action"],  # 7L arm + 7R arm + 2 gripper cmds (IK result)
            },
            "obj_init": {
                "dtype": "float32",
                "shape": (6,),
                "names": ["obj_init"],  # mug pos (3) + plate pos (3)
            },
        },
        image_writer_threads=10,
        image_writer_processes=5,
    )
else:
    print("Load from previous dataset")
    dataset = LeRobotDataset(REPO_NAME, root=ROOT)

## Keyboard Control
See the full key bindings in the header cell above.

Pressing **Z** resets the environment and discards the current episode buffer.

**To receive the success signal:** release both grippers and move the arm upward above the mug after placing it on the plate.

### Now let's teleop our robot and collect data!

**To receive the success signal, you have to release the gripper and move upwards above the mug!**

In [ ]:
action = np.zeros(14)  # EEF delta action: [dpose_l(6), gripper_l, dpose_r(6), gripper_r]
episode_id = 0
record_flag = False  # Start recording when the robot starts moving

while PnPEnv.env.is_viewer_alive() and episode_id < NUM_DEMO:
    PnPEnv.step_env()

    if PnPEnv.env.loop_every(HZ=20):
        # Check if the episode is done
        done = PnPEnv.check_success()
        if done:
            dataset.save_episode()
            PnPEnv.reset(seed=SEED)
            episode_id += 1

        # Teleoperate: returns 14D EEF delta action + reset flag
        action, reset = PnPEnv.teleop_robot()

        if not record_flag and np.any(action != 0):
            record_flag = True
            print("Start recording")

        if reset:
            # Z key pressed: discard current episode and restart
            PnPEnv.reset(seed=SEED)
            dataset.clear_episode_buffer()
            record_flag = False

        # Grab images from all three cameras
        agent_image, top_image, side_image = PnPEnv.grab_image()
        agent_image = np.array(Image.fromarray(agent_image).resize((256, 256)))
        top_image   = np.array(Image.fromarray(top_image).resize((256, 256)))
        side_image  = np.array(Image.fromarray(side_image).resize((256, 256)))

        # Step: runs IK on EEF action, sets self.q (joint targets), returns current joint state
        joint_q  = PnPEnv.step(action)              # current joint state (16D)
        target_q = PnPEnv.q.astype(np.float32)      # IK-solved joint targets (16D)

        if record_flag:
            dataset.add_frame(
                {
                    "observation.images.agentview": agent_image,
                    "observation.images.topview":   top_image,
                    "observation.images.sideview":  side_image,
                    "observation.state": joint_q,   # float32 from get_joint_state()
                    "action":            target_q,  # float32 cast
                    "obj_init":          PnPEnv.obj_init_pose,
                },
                task=TASK_NAME,
            )
            print(joint_q)

        PnPEnv.render(teleop=True)

In [ ]:
PnPEnv.env.close_viewer()

In [ ]:
# Clean up the images folder
import shutil
shutil.rmtree(dataset.root / 'images')